# German Industrial Automation RAG Assistant

## Project goal
Build a German-language assistant that answers beginner questions about industrial control systems using a small set of technical textbook pages.

## How it works
1. Load five German textbook pages.
2. Split their text into passages and create multilingual embeddings.
3. Find passages relevant to a question.
4. Ask Gemini to answer from those passages and cite them.
5. Check whether it answers supported questions and abstains when the sources do not contain an answer.

## Source attribution
Technical source: *Regelungstechnik – Das interaktive Lehrbuch* by Heiko Peuscher  
License: [Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

Pages used:
- [Regelungsproblem](https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/regelungsproblem)
- [Zweipunktregler](https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/zweipunktregler)
- [Dreipunktregler](https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/dreipunktregler)
- [PID-Regelung](https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/pidregelung)
- [Gütekriterien](https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/guetekriterien)

Changes: I extracted the article text from the web pages, split it into passages for search, and created AI-generated answers from retrieved passages. The generated answers are not written or reviewed by the textbook author.

This is an independent learning project and is not endorsed by the author.

## Scope and limitations
This demo searches only five pages and has been checked with four hand-picked questions. It is a learning prototype, not a validated system for industrial operating or safety decisions.

In [34]:
!pip -q install beautifulsoup4 requests

import requests
from bs4 import BeautifulSoup

url = "https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/zweipunktregler"

response = requests.get(url, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

title = soup.find("h1").get_text(" ", strip=True)
text = soup.get_text(" ", strip=True)

print("Article:", title)
print(text[:1000])

Article: Der Zweipunktregler
Der Zweipunktregler: Funktionsweise und interaktive Beispiele Regelungstechnik Das interaktive Lehrbuch Regelungstechnik Klassischer Reglerentwurf Zweipunktregler Suche Anpassen Wer bist Du? Schüler:in Student:in Berufstätig Lehrkraft Inhaltstiefe Schule Hochschule Formelzeichen DIN (y, x) Lehrbuch (u, y) Darstellung ☀️ Hell 🌙 Dunkel Navigation Regelungstechnik Wozu Regelungstechnik? Beispiel: Backofen Beispiel: Tempomat Beispiel: Helligkeit Das Regelungsproblem Gütekriterien Klassischer Reglerentwurf Zweipunktregler Dreipunktregler PID-Regelung Der P-Regler Der PI-Regler Wendetangente Analyse: LZI-Regelkreis Gegenkopplung Stationäre Genauigkeit Stabilitätsanalyse Charakteristische Gleichung und Kritischer Punkt Spezielles Nyquist-Kriterium Allgemeines Nyquist-Kriterium Stabilitätsreserve Zustandsregelung Zustandsbeobachter Systemtheorie & Simulation Systeme und Signale Definition & Darstellung Kontinuierlich vs. diskret Systemeigenschaften Sprung und Impul

In [2]:
print("main area found:", soup.find("main") is not None)
print("article area found:", soup.find("article") is not None)

main area found: True
article area found: False


In [3]:
main = soup.find("main")
article_text = main.get_text(" ", strip=True)

print(article_text[:1000])

Der Zweipunktregler Der Zweipunktregler eignet sich für Strecken mit einem Stellglied, das nur zwei verschiedene Zustände einnehmen kann (z. B. an / aus). Erforderliches Vorwissen ▼ Bevor Du diesen Artikel liest, solltest Du hiermit vertraut sein: Das Regelungsproblem Regelgesetz Der Zweipunktregler ermittelt die Stellgröße mithilfe einer Fallunterscheidung aus der Regeldifferenz: u y ( t ) = { u y m i n , wenn e ≤ 0 u y m a x , wenn e > 0 u y ( t ) = { u y min ​ , u y ma x ​ , ​ wenn e ≤ 0 wenn e > 0 ​ Die Stellgröße kann hier nur zwei verschiedene Werte annehmen, entweder den kleinstmöglichen (minimalen) u y m i n u y min ​ oder den größtmöglichen (maximalen) u y m a x u y ma x ​ . Beispiel: Backofen ▼ Erinnere Dich an das Beispiel des Backofens zurück, bei dem wir den konstanten Sollwert w = 180 w = 180 ° einstellen wollten. Das Heizelement kann nur entweder an sein ( u y = u y m a x = 1 u y = u y ma x ​ = 1 ) oder aus ( u y = u y m i n = 0 u y = u y min ​ = 0 ) — das passt gut zum 

In [4]:
source_pages = {
    "Regelungsproblem": "https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/regelungsproblem",
    "Zweipunktregler": "https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/zweipunktregler",
    "Dreipunktregler": "https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/dreipunktregler",
    "PID-Regelung": "https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/pidregelung",
    "Gütekriterien": "https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/guetekriterien"
}

documents = []

for source_name, page_url in source_pages.items():
    page_response = requests.get(page_url, timeout=30)
    page_response.raise_for_status()

    page_soup = BeautifulSoup(page_response.text, "html.parser")
    main_content = page_soup.find("main")
    page_text = main_content.get_text(" ", strip=True) if main_content else ""

    documents.append({
        "source": source_name,
        "url": page_url,
        "text": page_text
    })

print("Pages loaded:", len(documents))

for document in documents:
    print(document["source"], "| characters:", len(document["text"]))

Pages loaded: 5
Regelungsproblem | characters: 4370
Zweipunktregler | characters: 4809
Dreipunktregler | characters: 2171
PID-Regelung | characters: 2016
Gütekriterien | characters: 2155


In [5]:
first_text = documents[0]["text"]

chunk_size = 700
overlap = 100
first_chunks = []

start = 0

while start < len(first_text):
    end = start + chunk_size
    first_chunks.append(first_text[start:end])
    start = end - overlap

print("Source:", documents[0]["source"])
print("Original characters:", len(first_text))
print("Passages created:", len(first_chunks))
print("\nFirst passage:")
print(first_chunks[0])

Source: Regelungsproblem
Original characters: 4370
Passages created: 8

First passage:
Das Regelungsproblem Bevor wir Dir verraten, wie ein Regler funktioniert, solltest Du ein paar wichtige Fachbegriffe kennenlernen und die Herausforderung der Regelungstechnik in etwas allgemeinerer Form verstehen. — Nein, ich will gleich wissen, wie der Regler funktioniert! Der interaktive Signalflussplan unten baut sich nach und nach auf, wenn Du auf "weiter" klickst; Du kannst auch die Pfeiltasten benutzen. Ganz rechts siehst Du die Regelgröße y x y x — sie ist das Signal, das wir kontrollieren wollen, z.B. die Innentemperatur des Backofens. zurück weiter ??? Strecke Messglied w y x r z Den zugehörigen Sollwert siehst Du links; man bezeichnet ihn auch als Führungsgröße w w . Die Regelgröße


In [6]:
chunks = []

for document in documents:
    text = document["text"]
    start = 0
    chunk_number = 1

    while start < len(text):
        end = start + chunk_size
        passage = text[start:end]

        chunks.append({
            "source": document["source"],
            "url": document["url"],
            "chunk_number": chunk_number,
            "text": passage
        })

        start = end - overlap
        chunk_number += 1

print("Total passages:", len(chunks))
print("First passage source:", chunks[0]["source"])
print("First passage number:", chunks[0]["chunk_number"])
print("First passage text:", chunks[0]["text"][:250])

Total passages: 29
First passage source: Regelungsproblem
First passage number: 1
First passage text: Das Regelungsproblem Bevor wir Dir verraten, wie ein Regler funktioniert, solltest Du ein paar wichtige Fachbegriffe kennenlernen und die Herausforderung der Regelungstechnik in etwas allgemeinerer Form verstehen. — Nein, ich will gleich wissen, wie 


In [7]:
!pip -q install sentence-transformers

from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

passage_texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = model.encode_document(
    passage_texts,
    show_progress_bar=True
)

print("Number of passages:", len(passage_texts))
print("Embedding shape:", chunk_embeddings.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of passages: 29
Embedding shape: (29, 384)


In [8]:
question = "Wie funktioniert ein Zweipunktregler?"

query_embedding = model.encode_query(question)
scores = model.similarity(query_embedding, chunk_embeddings)[0]

top_indices = scores.argsort(descending=True)[:3]

for rank, index in enumerate(top_indices, start=1):
    chunk = chunks[int(index)]
    score = scores[index].item()

    print(f"\nResult {rank} | Similarity: {score:.3f}")
    print("Source:", chunk["source"])
    print("Passage number:", chunk["chunk_number"])
    print(chunk["text"][:350])


Result 1 | Similarity: 0.788
Source: Zweipunktregler
Passage number: 1
Der Zweipunktregler Der Zweipunktregler eignet sich für Strecken mit einem Stellglied, das nur zwei verschiedene Zustände einnehmen kann (z. B. an / aus). Erforderliches Vorwissen ▼ Bevor Du diesen Artikel liest, solltest Du hiermit vertraut sein: Das Regelungsproblem Regelgesetz Der Zweipunktregler ermittelt die Stellgröße mithilfe einer Fallunter

Result 2 | Similarity: 0.729
Source: Dreipunktregler
Passage number: 1
Der Dreipunktregler Der Dreipunktregler eignet sich für Strecken mit einem Stellglied, das drei verschiedene Zustände einnehmen kann. Er hat große Ähnlichkeit mit dem Zweipunktregler , der nur zwei Zustände unterscheidet (z.B. an / aus). Erforderliches Vorwissen ▼ Bevor Du diesen Artikel liest, solltest Du hiermit vertraut sein: Das Regelungsproble

Result 3 | Similarity: 0.544
Source: Regelungsproblem
Passage number: 1
Das Regelungsproblem Bevor wir Dir verraten, wie ein Regler funktioniert, solltes

In [9]:
%pip -q install -U google-genai

from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

print("Gemini client is ready")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
Gemini client is ready


In [10]:
context_parts = []

for rank, index in enumerate(top_indices, start=1):
    chunk = chunks[int(index)]

    context_parts.append(
        f"[Quelle {rank}: {chunk['source']}, Passage {chunk['chunk_number']}]\n"
        f"{chunk['text']}"
    )

context = "\n\n".join(context_parts)

prompt = f"""
Du bist ein deutschsprachiger Lernassistent für industrielle Automatisierung.

Beantworte die Frage in einfachen Worten und verwende nur die Informationen
aus den Quellenpassagen unten. Erfinde keine zusätzlichen Fakten.
Wenn die Quellen die Frage nicht ausreichend beantworten, sage das klar.
Setze Quellenhinweise wie [Quelle 1] direkt hinter die passenden Aussagen.

Frage:
{question}

Quellenpassagen:
{context}
"""

interaction = client.interactions.create(
    model="gemini-3.8-flash",
    input=prompt
)

print("Antwort:")
print(interaction.output_text)

print("\nQuellen:")
for rank, index in enumerate(top_indices, start=1):
    chunk = chunks[int(index)]
    print(f"[Quelle {rank}] {chunk['source']}: {chunk['url']}")

InternalServerError: Error code: 503 - {'error': {'message': 'gemini-3.8-flash is currently experiencing high demand, spikes in demand are usually temporary. Please try again later.', 'code': 'service_unavailable'}}

In [11]:
interaction = client.interactions.create(
    model="gemini-3.5-flash-lite",
    input=prompt
)

print("Antwort:")
print(interaction.output_text)

Antwort:
Ein Zweipunktregler funktioniert mithilfe einer Fallunterscheidung aus der Regeldifferenz [Quelle 1]. 

Er eignet sich für Strecken, bei denen das Stellglied nur zwei verschiedene Zustände einnehmen kann, wie zum Beispiel „an“ und „aus“ [Quelle 1]. Dabei kann die Stellgröße nur zwei Werte annehmen: entweder den kleinstmöglichen oder den größtmöglichen Wert [Quelle 1]. 

Genau bedeutet das:
* Ist die Regeldifferenz kleiner oder gleich null ($e \le 0$), wird der kleinste Wert der Stellgröße gewählt [Quelle 1].
* Ist die Regeldifferenz größer als null ($e > 0$), wird der größte Wert der Stellgröße gewählt [Quelle 1].


In [12]:
print("Sources used:")

for rank, index in enumerate(top_indices, start=1):
    chunk = chunks[int(index)]
    print(
        f"[Quelle {rank}] {chunk['source']} — "
        f"Passage {chunk['chunk_number']} — {chunk['url']}"
    )

Sources used:
[Quelle 1] Zweipunktregler — Passage 1 — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/zweipunktregler
[Quelle 2] Dreipunktregler — Passage 1 — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/dreipunktregler
[Quelle 3] Regelungsproblem — Passage 1 — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/regelungsproblem


In [13]:
retrieval_checks = [
    {
        "question": "Wie funktioniert ein Zweipunktregler?",
        "expected_source": "Zweipunktregler"
    },
    {
        "question": "Welche Zustände hat ein Dreipunktregler?",
        "expected_source": "Dreipunktregler"
    },
    {
        "question": "Wie funktioniert ein PID-Regler?",
        "expected_source": "PID-Regelung"
    },
    {
        "question": "Welche Kriterien beurteilen die Qualität eines Reglers?",
        "expected_source": "Gütekriterien"
    }
]

correct_count = 0

for item in retrieval_checks:
    query_vector = model.encode_query(item["question"])
    scores = model.similarity(query_vector, chunk_embeddings)[0]

    best_index = int(scores.argmax().item())
    found_source = chunks[best_index]["source"]
    is_correct = found_source == item["expected_source"]

    correct_count += is_correct

    print("Question:", item["question"])
    print("Expected:", item["expected_source"])
    print("Found:", found_source)
    print("Correct:", is_correct)
    print()

print(f"Top-1 retrieval score: {correct_count}/{len(retrieval_checks)}")

Question: Wie funktioniert ein Zweipunktregler?
Expected: Zweipunktregler
Found: Zweipunktregler
Correct: True

Question: Welche Zustände hat ein Dreipunktregler?
Expected: Dreipunktregler
Found: Dreipunktregler
Correct: True

Question: Wie funktioniert ein PID-Regler?
Expected: PID-Regelung
Found: PID-Regelung
Correct: True

Question: Welche Kriterien beurteilen die Qualität eines Reglers?
Expected: Gütekriterien
Found: Regelungsproblem
Correct: False

Top-1 retrieval score: 3/4


In [14]:
question_to_check = "Welche Kriterien beurteilen die Qualität eines Reglers?"

query_vector = model.encode_query(question_to_check)
scores = model.similarity(query_vector, chunk_embeddings)[0]
best_indices = scores.argsort(descending=True)[:3]

for rank, index in enumerate(best_indices, start=1):
    chunk = chunks[int(index)]

    print(f"\nMatch {rank} | Similarity: {scores[index].item():.3f}")
    print("Source:", chunk["source"])
    print(chunk["text"][:300])


Match 1 | Similarity: 0.680
Source: Regelungsproblem
n, d.h. er darf sich nicht von selbst aufschaukeln. Störunterdrückung : Störgrößen sollen möglichst wenig Einfluss auf die Regelgröße haben. Robustheit : Der Regler soll auch dann noch gut funktionieren, wenn sich die Eigenschaften der Regelstrecke ändern, z.B. durch Alterung oder Verschleiß. Dieser

Match 2 | Similarity: 0.643
Source: Gütekriterien
Gütekriterien zur Beurteilung eines Reglers Motivation Stell Dir vor, Du gibst einem Regelkreis einen sprungförmigen Sollwert-Verlauf vor und erhältst für vier verschiedene Regler-Einstellungen die folgenden Verläufe der Regelgröße. Wie die Aufgabe oben verdeutlicht hat, können sich Regelkreise in m

Match 3 | Similarity: 0.551
Source: Regelungsproblem
Das Regelungsproblem Bevor wir Dir verraten, wie ein Regler funktioniert, solltest Du ein paar wichtige Fachbegriffe kennenlernen und die Herausforderung der Regelungstechnik in etwas allgemeinerer Form verstehen. — Nein, ich will gleich w

In [30]:
def ask_assistant(question, top_k=3):
    query_vector = model.encode_query(question)
    scores = model.similarity(query_vector, chunk_embeddings)[0]
    selected_indices = scores.argsort(descending=True)[:top_k]

    context_parts = []
    selected_sources = []

    for rank, index in enumerate(selected_indices, start=1):
        chunk = chunks[int(index)]

        context_parts.append(
            f"[Quelle {rank}: {chunk['source']}, "
            f"Passage {chunk['chunk_number']}]\n{chunk['text']}"
        )
        selected_sources.append((rank, chunk))

    context = "\n\n".join(context_parts)

    prompt = f"""
Du bist ein deutschsprachiger Lernassistent für industrielle Automatisierung.

Beantworte die Frage in einfachen Worten und verwende ausschließlich die
Informationen aus den Quellenpassagen.

Prüfe alle Quellenpassagen. Wenn eine Passage die Antwort enthält, beantworte
die Frage direkt, auch wenn sie anders formuliert ist. Wenn nach mehreren
Punkten gefragt wird, liste alle passenden Punkte aus den Quellen auf.

Sage "Dazu finde ich in den bereitgestellten Quellen keine ausreichende
Information" nur, wenn keine der Passagen die Antwort enthält.
Setze Quellenhinweise wie [Quelle 1] direkt hinter die Aussagen, die sie stützen.

Frage:
{question}

Quellenpassagen:
{context}
"""

    interaction = client.interactions.create(
        model="gemini-3.5-flash-lite",
        input=prompt
    )

    print("Antwort:")
    print(interaction.output_text)

    print("\nQuellen:")
    for rank, chunk in selected_sources:
        print(f"[Quelle {rank}] {chunk['source']} — {chunk['url']}")

In [31]:
ask_assistant("Nenne die vier Gütekriterien zur Beurteilung eines Reglers.", top_k=7)

Antwort:
Dazu finde ich in den bereitgestellten Quellen keine ausreichende Information, da dort vier Gütekriterien genannt werden (Totzeit, Anregelzeit, Ausregelzeit und Überschwingweite) [Quelle 7], aber nicht nach vier *anderen* oder einer anderen bestimmten Anzahl gefragt wird, beziehungsweise die vier genannten in den Quellen aufgeführt sind. 

*(Hinweis: Falls die in [Quelle 7] genannten vier Gütekriterien gemeint sind, sind dies:)*
* Die Totzeit [Quelle 7]
* Die Anregelzeit [Quelle 7]
* Die Ausregelzeit [Quelle 7]
* Die Überschwingweite [Quelle 7]

Quellen:
[Quelle 1] Gütekriterien — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/guetekriterien
[Quelle 2] Dreipunktregler — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/dreipunktregler
[Quelle 3] Regelungsproblem — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/regelungsproblem
[Quelle 4] PID-Regelung — https://www.regelungstechnik.de/lehrbuch/regelungstechn

In [17]:
ask_assistant("Welche Wartungsintervalle gelten für eine CNC-Fräsmaschine?")

Antwort:
Dazu finde ich in den bereitgestellten Quellen keine ausreichende Information.

Quellen:
[Quelle 1] Regelungsproblem — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/regelungsproblem
[Quelle 2] Gütekriterien — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/guetekriterien
[Quelle 3] Zweipunktregler — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/zweipunktregler


In [18]:
ask_assistant("Nenne die vier Gütekriterien zur Beurteilung eines Reglers.")

Antwort:
Dazu finde ich in den bereitgestellten Quellen keine ausreichende Information.

Quellen:
[Quelle 1] Gütekriterien — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/guetekriterien
[Quelle 2] Dreipunktregler — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/dreipunktregler
[Quelle 3] Regelungsproblem — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/regelungsproblem


In [19]:
question_to_check = "Nenne die vier Gütekriterien zur Beurteilung eines Reglers."

query_vector = model.encode_query(question_to_check)
scores = model.similarity(query_vector, chunk_embeddings)[0]
best_indices = scores.argsort(descending=True)[:5]

for rank, index in enumerate(best_indices, start=1):
    chunk = chunks[int(index)]

    print(
        f"\nMatch {rank} | Similarity: {scores[index].item():.3f} | "
        f"Source: {chunk['source']} | Passage: {chunk['chunk_number']}"
    )
    print(chunk["text"][:500])


Match 1 | Similarity: 0.726 | Source: Gütekriterien | Passage: 1
Gütekriterien zur Beurteilung eines Reglers Motivation Stell Dir vor, Du gibst einem Regelkreis einen sprungförmigen Sollwert-Verlauf vor und erhältst für vier verschiedene Regler-Einstellungen die folgenden Verläufe der Regelgröße. Wie die Aufgabe oben verdeutlicht hat, können sich Regelkreise in mehrerer Weise hinsichtlich der erreichten Regelgüte unterscheiden. Man benötigt daher objektive Kriterien, mit denen man verschiedene Reglereinstellungen bewerten und untereinander vergleichen kann. W

Match 2 | Similarity: 0.525 | Source: Dreipunktregler | Passage: 1
Der Dreipunktregler Der Dreipunktregler eignet sich für Strecken mit einem Stellglied, das drei verschiedene Zustände einnehmen kann. Er hat große Ähnlichkeit mit dem Zweipunktregler , der nur zwei Zustände unterscheidet (z.B. an / aus). Erforderliches Vorwissen ▼ Bevor Du diesen Artikel liest, solltest Du hiermit vertraut sein: Das Regelungsproblem Zweipunktregl

In [20]:
question_to_check = "Nenne die vier Gütekriterien zur Beurteilung eines Reglers."

query_vector = model.encode_query(question_to_check)
all_scores = model.similarity(query_vector, chunk_embeddings)[0]

for index, chunk in enumerate(chunks):
    if chunk["source"] == "Gütekriterien":
        print(
            f"\nPassage {chunk['chunk_number']} | "
            f"Similarity: {all_scores[index].item():.3f}"
        )
        print(chunk["text"])


Passage 1 | Similarity: 0.726
Gütekriterien zur Beurteilung eines Reglers Motivation Stell Dir vor, Du gibst einem Regelkreis einen sprungförmigen Sollwert-Verlauf vor und erhältst für vier verschiedene Regler-Einstellungen die folgenden Verläufe der Regelgröße. Wie die Aufgabe oben verdeutlicht hat, können sich Regelkreise in mehrerer Weise hinsichtlich der erreichten Regelgüte unterscheiden. Man benötigt daher objektive Kriterien, mit denen man verschiedene Reglereinstellungen bewerten und untereinander vergleichen kann. Wichtige Gütekriterien Besonders gebräuchlich sind einige Gütekriterien, die man von der Reaktion der Regelgröße auf einen Sprung der Führungsgröße ablesen kann. Die Abbildung unten zeigt einen solchen

Passage 2 | Similarity: 0.386
 Regelgröße auf einen Sprung der Führungsgröße ablesen kann. Die Abbildung unten zeigt einen solchen Sollwertsprung (blau) und dazu einen möglichen Verlauf der Regelgröße (grün). Da der Istwert den Sollwert in der Praxis nie perfekt erre

In [21]:
chunk_size = 1200
overlap = 150
chunks = []

for document in documents:
    text = document["text"]
    start = 0
    chunk_number = 1

    while start < len(text):
        end = start + chunk_size
        passage = text[start:end]

        chunks.append({
            "source": document["source"],
            "url": document["url"],
            "chunk_number": chunk_number,
            "text": passage
        })

        start = end - overlap
        chunk_number += 1

passage_texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = model.encode_document(
    passage_texts,
    show_progress_bar=True
)

print("Passage size:", chunk_size, "characters")
print("Total passages:", len(chunks))
print("Embedding shape:", chunk_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Passage size: 1200 characters
Total passages: 18
Embedding shape: (18, 384)


In [22]:
ask_assistant("Nenne die vier Gütekriterien zur Beurteilung eines Reglers.")

Antwort:
Dazu finde ich in den bereitgestellten Quellen keine ausreichende Information.

Quellen:
[Quelle 1] Gütekriterien — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/guetekriterien
[Quelle 2] Dreipunktregler — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/dreipunktregler
[Quelle 3] Regelungsproblem — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/regelungsproblem


In [23]:
question_to_check = "Nenne die vier Gütekriterien zur Beurteilung eines Reglers."

query_vector = model.encode_query(question_to_check)
scores = model.similarity(query_vector, chunk_embeddings)[0]
ranked_indices = scores.argsort(descending=True)

print("Top three passages the assistant receives:")

for rank, index in enumerate(ranked_indices[:3], start=1):
    chunk = chunks[int(index)]
    print(
        f"\nMatch {rank} | Score: {scores[index].item():.3f} | "
        f"{chunk['source']} | Passage {chunk['chunk_number']}"
    )
    print(chunk["text"])

print("\nAll passages from Gütekriterien:")

for index, chunk in enumerate(chunks):
    if chunk["source"] == "Gütekriterien":
        print(
            f"\nPassage {chunk['chunk_number']} | "
            f"Score: {scores[index].item():.3f}"
        )
        print(chunk["text"])

Top three passages the assistant receives:

Match 1 | Score: 0.726 | Gütekriterien | Passage 1
Gütekriterien zur Beurteilung eines Reglers Motivation Stell Dir vor, Du gibst einem Regelkreis einen sprungförmigen Sollwert-Verlauf vor und erhältst für vier verschiedene Regler-Einstellungen die folgenden Verläufe der Regelgröße. Wie die Aufgabe oben verdeutlicht hat, können sich Regelkreise in mehrerer Weise hinsichtlich der erreichten Regelgüte unterscheiden. Man benötigt daher objektive Kriterien, mit denen man verschiedene Reglereinstellungen bewerten und untereinander vergleichen kann. Wichtige Gütekriterien Besonders gebräuchlich sind einige Gütekriterien, die man von der Reaktion der Regelgröße auf einen Sprung der Führungsgröße ablesen kann. Die Abbildung unten zeigt einen solchen Sollwertsprung (blau) und dazu einen möglichen Verlauf der Regelgröße (grün). Da der Istwert den Sollwert in der Praxis nie perfekt erreicht, definiert man üblicherweise ein Toleranzband um den konstanten

In [24]:
ask_assistant(
    "Nenne die vier Gütekriterien zur Beurteilung eines Reglers.",
    top_k=5
)


Antwort:
Dazu finde ich in den bereitgestellten Quellen keine ausreichende Information.

Quellen:
[Quelle 1] Gütekriterien — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/guetekriterien
[Quelle 2] Dreipunktregler — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/dreipunktregler
[Quelle 3] Regelungsproblem — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/regelungsproblem
[Quelle 4] PID-Regelung — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/pidregelung
[Quelle 5] Regelungsproblem — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/regelungsproblem


In [25]:
question_to_check = "Nenne die vier Gütekriterien zur Beurteilung eines Reglers."

query_vector = model.encode_query(question_to_check)
scores = model.similarity(query_vector, chunk_embeddings)[0]
ranked_indices = scores.argsort(descending=True)[:10]

for rank, index in enumerate(ranked_indices, start=1):
    chunk = chunks[int(index)]

    print(
        f"Rank {rank} | Score: {scores[index].item():.3f} | "
        f"Source: {chunk['source']} | Passage: {chunk['chunk_number']}"
    )

Rank 1 | Score: 0.726 | Source: Gütekriterien | Passage: 1
Rank 2 | Score: 0.525 | Source: Dreipunktregler | Passage: 1
Rank 3 | Score: 0.455 | Source: Regelungsproblem | Passage: 1
Rank 4 | Score: 0.423 | Source: PID-Regelung | Passage: 1
Rank 5 | Score: 0.416 | Source: Regelungsproblem | Passage: 2
Rank 6 | Score: 0.412 | Source: Zweipunktregler | Passage: 1
Rank 7 | Score: 0.400 | Source: Gütekriterien | Passage: 2
Rank 8 | Score: 0.360 | Source: PID-Regelung | Passage: 2
Rank 9 | Score: 0.352 | Source: Regelungsproblem | Passage: 4
Rank 10 | Score: 0.332 | Source: Regelungsproblem | Passage: 3


In [26]:
ask_assistant(
    "Nenne die vier Gütekriterien zur Beurteilung eines Reglers.",
    top_k=7
)

Antwort:
Dazu finde ich in den bereitgestellten Quellen keine ausreichende Information. Die Quellen nennen zwar vier Messgrößen beziehungsweise Eigenschaften zur Beurteilung (Totzeit, Anregelzeit, Ausregelzeit und Überschwingweite) [Quelle 7], bezeichnen diese im Text jedoch nicht explizit als „die vier Gütekriterien“.

Quellen:
[Quelle 1] Gütekriterien — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/guetekriterien
[Quelle 2] Dreipunktregler — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/dreipunktregler
[Quelle 3] Regelungsproblem — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/regelungsproblem
[Quelle 4] PID-Regelung — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/pidregelung
[Quelle 5] Regelungsproblem — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/regelungsproblem
[Quelle 6] Zweipunktregler — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/kla

In [27]:
prompt = f"""
Du bist ein deutschsprachiger Lernassistent für industrielle Automatisierung.

Beantworte die Frage direkt und in einfachen Worten. Verwende ausschließlich
Informationen aus den Quellenpassagen. Wenn eine Passage die Antwort enthält,
beantworte die Frage auch dann, wenn sie anders formuliert ist als die Frage.
Wenn die Frage nach einer Liste fragt, nenne alle passenden Punkte aus den Quellen.

Setze Quellenhinweise wie [Quelle 1] direkt hinter die Aussagen, die sie stützen.
Sage nur dann "Dazu finde ich in den bereitgestellten Quellen keine ausreichende
Information", wenn die Quellen die Antwort tatsächlich nicht enthalten.
Widerspreche dieser Aussage nicht, indem Du danach doch eine Antwort gibst.

Frage:
{question}

Quellenpassagen:
{context}
"""

In [28]:
ask_assistant(
    "Nenne die vier Gütekriterien zur Beurteilung eines Reglers.",
    top_k=7
)

Antwort:
Dazu finde ich in den bereitgestellten Quellen keine ausreichende Information.

Quellen:
[Quelle 1] Gütekriterien — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/guetekriterien
[Quelle 2] Dreipunktregler — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/dreipunktregler
[Quelle 3] Regelungsproblem — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/regelungsproblem
[Quelle 4] PID-Regelung — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/pidregelung
[Quelle 5] Regelungsproblem — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/regelungsproblem
[Quelle 6] Zweipunktregler — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/zweipunktregler
[Quelle 7] Gütekriterien — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/guetekriterien


In [32]:
ask_assistant(
    "Nenne die vier Gütekriterien zur Beurteilung eines Reglers.",
    top_k=7
)

Antwort:
Die vier Gütekriterien zur Beurteilung eines Reglers sind:

1. **Die Totzeit:** Der Zeitraum nach einem Führungsgrößensprung, in dem die Regelgröße noch keine Reaktion zeigt [Quelle 7].
2. **Die Anregelzeit:** Die Dauer, bis die Regelgröße nach dem Sprung erstmals in das gewünschte Toleranzintervall eintritt [Quelle 7].
3. **Die Ausregelzeit:** Die Dauer des Einschwingvorgangs, bis die Regelgröße dauerhaft im Toleranzintervall bleibt und es nicht mehr verlässt [Quelle 7].
4. **Die Überschwingweite:** Sie beschreibt, wie weit die Regelgröße den gewünschten Sollwert im Maximum überschritten hat (wird üblicherweise in Prozent der Sollsprunghöhe angegeben) [Quelle 7].

Quellen:
[Quelle 1] Gütekriterien — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/guetekriterien
[Quelle 2] Dreipunktregler — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/dreipunktregler
[Quelle 3] Regelungsproblem — https://www.regelungstechnik.de/lehrbuch/regel

In [33]:
ask_assistant(
    "Welche Wartungsintervalle gelten für eine CNC-Fräsmaschine?",
    top_k=3
)

Antwort:
Dazu finde ich in den bereitgestellten Quellen keine ausreichende Information.

Quellen:
[Quelle 1] Zweipunktregler — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/zweipunktregler
[Quelle 2] Gütekriterien — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/einfuehrung/guetekriterien
[Quelle 3] Zweipunktregler — https://www.regelungstechnik.de/lehrbuch/regelungstechnik/klassischeregler/zweipunktregler


## Initial manual evaluation

I tested four questions against the assistant's source material.

| Test | Expected behavior | Result |
|---|---|---|
| Explain a Zweipunktregler | Give a source-supported explanation | Pass |
| Compare Zwei- and Dreipunktregler | Explain the difference with citations | Pass |
| Name the four Gütekriterien | List all four with a citation | Pass |
| Ask about CNC maintenance intervals | Say the sources do not provide enough information | Pass |

These are four hand-picked checks, not a large benchmark. They show that the current examples work, but do not prove general accuracy.